# Subsetting the Larger Data Files

The base map is a map of Canada with Census Division boundaries drawn on it. The original data is a shapefile from the 2021 Statistics Canada Census, which can be found [here](https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index2021-eng.cfm?year=21).

To get the same file, download with these options
- Cartographic Boundary Files (CBF)
- Administrative Boundaries --> Census divisions (don't check any other boxes for geography type)
- Shapefile (.shp)

Note, the following cells can only be run on Derek's computer, because the files are too large to commit. Skip to the cell after the title that says "Subsetted Data".

## Shapefile

#### CLI commands used to simplify the shapefile

(Run all of the following commands in the root of the repository).

To install mapshaper for shapefile compression, first install nodejs which will let us put mapshaper in the conda env:

```bash
conda install -c conda-forge nodejs
```
then install it in the conda env:

```bash
npm install -g mapshaper
```
then, perform the compression:

```bash
mapshaper ../original_data/shapefiles/lcd_000b21a_e.shp -simplify 5% -o data/geojson/lcd_000b21a_e_simplified_5percent.geojson
```

## Immigration dataset (CSV)

Link: <https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=9810030701>

First let's explore the entire CSV, which has every geography type. These are:

- Canada
- Province or territory
- Census division
- Census subdivision

## Subsetted Data

Everything below is subsetted (files are committed to the repo) and these cells can be run by anyone on the team:

In [11]:
import geopandas as gpd
import pandas

# Read the 4.1 MB simplified file
gdf = gpd.read_file("../data/raw/geojson/lcsd000b21a_e_simplified_1percent.geojson")

# Inspect available columns
print(gdf.columns)

Index(['CSDUID', 'DGUID', 'CSDNAME', 'CSDTYPE', 'LANDAREA', 'PRUID',
       'geometry'],
      dtype='object')


In [12]:
gdf.head()

,CSDUID,DGUID,CSDNAME,CSDTYPE,LANDAREA,PRUID,geometry
0,1001101,2021A00051001101,"Division No. 1, Subd. V",SNO,870.8928,10,"POLYGON ((9013296.10857 2066441.22, 9012550.37..."
1,1001105,2021A00051001105,Portugal Cove South,T,1.0770,10,"POLYGON ((9001007.67714 2049580.86, 9001538.83..."
2,1001113,2021A00051001113,Trepassey,T,54.2130,10,"POLYGON ((8997089.68 2047620.21714, 8992293.85..."
3,1001120,2021A00051001120,St. Shott's,T,1.0729,10,"POLYGON ((8984658.56571 2028232.80571, 8985460..."
4,1001124,2021A00051001124,"Division No. 1, Subd. U",SNO,742.3781,10,"POLYGON ((8995279.39429 2119585.06, 8992653.23..."


In [13]:
# # List of subdivision types to EXCLUDE (too small or trivial)
# exclude_types = [
#     "CC", "CG", "CN", "DM", "FD", "GR", "ID", "IGD", "IM", "IRI", "LGD",
#     "NH", "NL", "NO", "NV", "RCR", "RDA", "S-É", "SA", "SC", "SET", "SG",
#     "SNO", "SV", "TAL", "TC", "TI", "TK", "TL", "TWL", "VC", "VK", "VN"
# ]

# # Filter the GeoDataFrame to remove small subdivisions
# gdf = gdf[~gdf["CSDTYPE"].isin(exclude_types)]

Check if there's null rows. Null rows will break the map painting

In [14]:
null_columns = gdf.columns[gdf.isnull().any()]
print(gdf[null_columns].isnull().sum())
print(gdf[gdf.isnull().any(axis=1)])

geometry    422
dtype: int64
       CSDUID             DGUID          CSDNAME CSDTYPE  LANDAREA PRUID  \
389   1101050  2021A00051101050         Morell 2     IRI    0.7828    11   
405   1102030  2021A00051102030    Rocky Point 3     IRI    0.0424    11   
447   1103027  2021A00051103027    Abram-Village      RM    1.3599    11   
455   1103050  2021A00051103050        Northport      RM    1.7280    11   
459   1103057  2021A00051103057        St. Louis      RM    0.6846    11   
...       ...               ...              ...     ...       ...   ...   
5046  5957813  2021A00055957813       Lower Post     S-É    0.1506    59   
5050  5959805  2021A00055959805         Fontas 1     IRI    0.1287    59   
5052  5959809  2021A00055959809        Kahntah 3     IRI    0.0825    59   
5058  6001008  2021A00056001008       Carcross 4      SG    0.5774    60   
5121  6105004  2021A00056105004  Salt Plains 195     IRI    0.3494    61   

     geometry  
389      None  
405      None  
447      N

In [15]:
columns_to_keep = ['DGUID', 'CSDUID', 'CSDNAME', 'geometry']
gdf = gdf[columns_to_keep]

In [16]:
gdf.head()

,DGUID,CSDUID,CSDNAME,geometry
0,2021A00051001101,1001101,"Division No. 1, Subd. V","POLYGON ((9013296.10857 2066441.22, 9012550.37..."
1,2021A00051001105,1001105,Portugal Cove South,"POLYGON ((9001007.67714 2049580.86, 9001538.83..."
2,2021A00051001113,1001113,Trepassey,"POLYGON ((8997089.68 2047620.21714, 8992293.85..."
3,2021A00051001120,1001120,St. Shott's,"POLYGON ((8984658.56571 2028232.80571, 8985460..."
4,2021A00051001124,1001124,"Division No. 1, Subd. U","POLYGON ((8995279.39429 2119585.06, 8992653.23..."


In [17]:
# Confirms the bounds are in meters. Not what we want!
print(gdf.total_bounds) 

[3689603.71714289  664795.9514286  9015464.8171429  5242009.10571432]


In [18]:
# 1) Force the correct CRS (EPSG:3347) 
#    This does not transform coordinates; it just sets the label to the right one.
gdf.crs = "EPSG:3347"

# 2) Now convert/transform from EPSG:3347 (meters) to EPSG:4326 (lat/lon)
gdf_latlon = gdf.to_crs(epsg=4326)

# Check the bounds now
print(gdf_latlon.total_bounds)
# Should be something like: [-141, 41, -52, 83] for Canada

[-141.01807316   41.72804164  -52.6275265    83.13654909]


In [19]:
# EPSG:4326 confirmed. Good!
gdf_latlon.crs 

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [20]:
gdf_latlon = gdf_latlon[gdf_latlon.geometry.notnull()]

In [21]:
import json

# Convert to a GeoJSON dictionary
geojson_data = json.loads(gdf_latlon.to_json())

# Assign "id" to each feature based on CDUID (so Plotly can match locations)
for feature in geojson_data["features"]:
    feature["id"] = feature["properties"]["CSDUID"]

In [22]:
# with open("output.geojson", "w") as f:
#     json.dump(geojson_data, f, indent=2)


In [23]:
import plotly.express as px
# Create a basic choropleth map
fig = px.choropleth_mapbox(
    gdf_latlon,
    geojson=geojson_data,
    locations="CSDUID",
    featureidkey="properties.CSDUID",
    color_discrete_sequence=["blue"],
    mapbox_style="carto-positron",
    center={"lat": 56, "lon": -106},
    zoom=3,
    opacity=0.7
)
fig.update_traces(marker_line_width=0.5, marker_line_color='black')
fig.show()

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\2905693242.py:3: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.choropleth_mapbox(


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# import plotly.express as px

# fig = px.choropleth_map(
#     gdf_latlon,
#     geojson=geojson_data,
#     locations='CSDUID',
#     color_discrete_sequence=["blue"],
#     map_style="open-street-map",  # or "carto-positron"
#     center={"lat": 56, "lon": -106},
#     zoom=4, 
#     opacity=0.7
# )
# fig.update_traces(marker_line_width=0.5, marker_line_color='black')
# fig.show()

In [ ]:
# import plotly.express as px

# fig = px.choropleth_mapbox(
#     gdf_latlon,
#     geojson=geojson_data,
#     locations="CSDUID",

#     # Remove fill color
#     color=None,

#     # Show Census Division names on hover
#     hover_name="CSDNAME",

#     # Map settings
#     mapbox_style="open-street-map",
#     center={"lat": 56, "lon": -106},
#     zoom=3,
#     opacity=0.7
# )

# # Make borders visible, remove legend, allow zoom on scroll
# fig.update_traces(marker_line_width=0.5, marker_line_color="black", showscale=False)

# # Remove white margins & enable zooming with scroll wheel
# fig.update_layout(
#     margin={"r": 0, "t": 0, "l": 0, "b": 0},
#     dragmode="zoom",
#     uirevision=True
# )

# # Enable zoom on scroll
# fig.show(config={"scrollZoom": True})

In [ ]:
# import plotly.express as px

# fig = px.choropleth_mapbox(
#     gdf_latlon,
#     geojson=geojson_data,
#     locations="CSDUID",
#     # Set fully transparent fill
#     color_discrete_sequence=["rgba(0,0,0,0)"],  # Invisible fill
#     # Show Census Division names on hover
#     hover_name="CSDNAME",
#     hover_data={"CSDUID": False},  # Removes CDUID from hover text
#     # Use an open-source tile style
#     mapbox_style="open-street-map",
#     center={"lat": 56, "lon": -106},
#     zoom=3,
#     opacity=1,  # Fully opaque lines (no transparency)
# )

# # Increase boundary thickness & remove legend
# fig.update_traces(
#     marker_line_width=1,  # Thicker boundary lines
#     marker_line_color="black",  # Outline color
#     showscale=False,  # Removes legend
# )

# # Remove white borders & enable zooming
# fig.update_layout(
#     margin={"r": 0, "t": 0, "l": 0, "b": 0}, dragmode="zoom", uirevision=True
# )

# # Enable zoom on scroll
# fig.show(config={"scrollZoom": True})

# Code for subsetting immigration data for BC census Subdivision usage

This cell below is for subsetting dataset. It should not be run repeatedly after subsetting.

In [46]:
import pandas as pd

chunk_size = 100000  # Adjust based on memory capacity
csv_path = "../data/raw/immigration_data/immigration_stats.csv"
csv_output_path = "../data/raw/immigration_data/immigration_stats_census_subdivisions.csv"

# Open an output file to write filtered data with UTF-8 encoding
with open(csv_output_path, "w", newline="", encoding="utf-8") as output_file:
    first_chunk = True  # To write headers only once

    for chunk in pd.read_csv(csv_path, chunksize=chunk_size, encoding="utf-8"):
        df_cd_chunk = chunk[chunk["DGUID"].astype(str).str.match(r"2021A0005\d{7}$", na=False)]
        
        # Append to CSV, writing headers only for the first chunk
        df_cd_chunk.to_csv(output_file, index=False, header=first_chunk, mode="a", encoding="utf-8")
        first_chunk = False  # Ensure headers are not written again

print(f"Filtered Census Division data saved to: {csv_output_path}")

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\3351651332.py:11: DtypeWarning:

Columns (8,10,12,14,16,18,20,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\3351651332.py:11: DtypeWarning:

Columns (8,10,12,14,16,18,20,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\3351651332.py:11: DtypeWarning:

Columns (8,10,12,14,16,18,20,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\3351651332.py:11: DtypeWarning:

Columns (8,10,12,14,16,18,20,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\3351651332.py:11: DtypeWarning:

Columns (8,10,12,14,16,18,20,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=Fal

Filtered Census Division data saved to: ../data/raw/immigration_data/immigration_stats_census_subdivisions.csv


In [44]:
df = pd.read_csv("../data/raw/immigration_data/immigration_stats_census_subdivisions.csv")

C:\Users\jinxi\AppData\Local\Temp\ipykernel_512424\1914775247.py:1: DtypeWarning:

Columns (8,10,12,14,16,18,20,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.



In [45]:
df[df["GEO"]=="Vancouver"]

,REF_DATE,GEO,DGUID,Age (8D),Gender (3),Place of birth (290),Coordinate,Immigrant status and period of immigration (11):Total - Immigrant status and period of immigration[1],Symbol,Immigrant status and period of immigration (11):Non-immigrants[2],...,Immigrant status and period of immigration (11):2001 to 2010[7],Symbol.6,Immigrant status and period of immigration (11):2011 to 2021[8],Symbol.7,Immigrant status and period of immigration (11):2011 to 2015[9],Symbol.8,Immigrant status and period of immigration (11):2016 to 2021[10],Symbol.9,Immigrant status and period of immigration (11):Non-permanent residents[11],Symbol.10


In [ ]:
df_immi_birth_place = df[(df["Age (8D)"] == "Total - Age") & (df["Gender (3)"] == "Total - Gender")]
df_immi_birth_place = df_immi_birth_place[["GEO", "DGUID", "Place of birth (290)", "Immigrant status and period of immigration (11):Total - Immigrant status and period of immigration[1]"]]
df_immi_birth_place.rename(
    columns={
        "Place of birth (290)": "Birthplace",
        "Immigrant status and period of immigration (11):Total - Immigrant status and period of immigration[1]": "Count"
    },
    inplace=True
)

total_count = df_immi_birth_place[df_immi_birth_place["Birthplace"]=="Total – Place of birth"]

total_count = total_count[["DGUID", "Count"]].set_index("DGUID")["Count"]

df_immi_birth_place["Proportion"] = df_immi_birth_place.apply(lambda row: row["Count"] / total_count[row["DGUID"]]*100, axis=1)

# df_immi_birth_place.set_index("DGUID", inplace=True)

# Unique birthplaces available
unique_birthplaces = df_immi_birth_place["Birthplace"].unique()

# Default birthplace to display (first in list)
default_birthplace = unique_birthplaces[0]

# Filter for the default birthplace
# df_filtered = df_immi_birth_place[df_immi_birth_place["Birthplace"] == default_birthplace]

# df_immi_birth_place = df_immi_birth_place.pivot(index=["GEO", "DGUID"], columns="birth", values="count")
df_immi_birth_place

C:\Users\jinxi\AppData\Local\Temp\ipykernel_419032\810414365.py:15: RuntimeWarning:

invalid value encountered in scalar divide



,GEO,DGUID,Birthplace,Count,Proportion
0,Elkford,2021A00055901003,Total – Place of birth,2745.0,100.000000
1,Elkford,2021A00055901003,Inside Canada,2525.0,91.985428
2,Elkford,2021A00055901003,Newfoundland and Labrador,75.0,2.732240
3,Elkford,2021A00055901003,Prince Edward Island,0.0,0.000000
4,Elkford,2021A00055901003,Nova Scotia,50.0,1.821494
...,...,...,...,...,...
5220285,Prophet River 4,2021A00055959810,Antarctica,0.0,NaN
5220286,Prophet River 4,2021A00055959810,Bouvet Island,0.0,NaN
5220287,Prophet River 4,2021A00055959810,French Southern Territories,0.0,NaN
5220288,Prophet River 4,2021A00055959810,Heard Island and McDonald Islands,0.0,NaN


In [ ]:
df_immi_only_birth_place = df

df_immi_only_birth_place = df_immi_only_birth_place[["GEO", "DGUID", "Gender (3)", "Age (8D)", "Place of birth (290)", "Immigrant status and period of immigration (11):Immigrants[3]"]]

df_immi_only_birth_place.rename(
    columns={
        "Place of birth (290)": "Birthplace",
        "Immigrant status and period of immigration (11):Immigrants[3]": "Count",
        "Gender (3)": "Gender",
        "Age (8D)": "Age"
    },
    inplace=True
)

total_count = df_immi_only_birth_place[df_immi_only_birth_place["Birthplace"] == "Total – Place of birth"]
total_count = total_count[["DGUID", "Gender", "Age", "Count"]].set_index(["DGUID", "Gender", "Age"])["Count"]

df_immi_only_birth_place = df_immi_only_birth_place[df_immi_only_birth_place["Count"] > 0]

df_immi_only_birth_place["Proportion"] = df_immi_only_birth_place.apply(
    lambda row: row["Count"] / total_count.get((row["DGUID"], row["Gender"], row["Age"]), 1) * 100, axis=1
)

df_immi_only_birth_place.dropna(subset=["Proportion"], inplace=True)

df_immi_only_birth_place["DGUID"] = df_immi_only_birth_place["DGUID"].astype("category")
df_immi_only_birth_place["Birthplace"] = df_immi_only_birth_place["Birthplace"].astype("category")
df_immi_only_birth_place["Gender"] = df_immi_only_birth_place["Gender"].astype("category")
df_immi_only_birth_place["Age"] = df_immi_only_birth_place["Age"].astype("category")

unique_birthplaces = df_immi_only_birth_place["Birthplace"].unique().tolist()
unique_genders = df_immi_only_birth_place["Gender"].unique().tolist()
unique_ages = df_immi_only_birth_place["Age"].unique().tolist()

df_immi_only_birth_place

C:\Users\jinxi\AppData\Local\Temp\ipykernel_419032\1846672369.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\jinxi\AppData\Local\Temp\ipykernel_419032\1846672369.py:19: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\jinxi\AppData\Local\Temp\ipykernel_419032\1846672369.py:19: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\jinxi\AppData\Local\Temp\ipykernel_419032\1846672369.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,GEO,DGUID,Gender,Age,Birthplace,Count,Proportion
0,Elkford,2021A00055901003,Total - Gender,Total - Age,Total – Place of birth,200.0,100.0
15,Elkford,2021A00055901003,Total - Gender,Total - Age,Outside Canada,200.0,100.0
16,Elkford,2021A00055901003,Total - Gender,Total - Age,Americas,45.0,22.5
17,Elkford,2021A00055901003,Total - Gender,Total - Age,North America,15.0,7.5
20,Elkford,2021A00055901003,Total - Gender,Total - Age,United States of America,15.0,7.5
...,...,...,...,...,...,...,...
5198655,Northern Rockies,2021A00055959007,Men+,65 to 74 years,United Kingdom,10.0,40.0
5198830,Northern Rockies,2021A00055959007,Women+,65 to 74 years,Total – Place of birth,15.0,100.0
5198845,Northern Rockies,2021A00055959007,Women+,65 to 74 years,Outside Canada,15.0,100.0
5198906,Northern Rockies,2021A00055959007,Women+,65 to 74 years,Europe,15.0,100.0


In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import json

def create_map(birthplace):
    df_filtered = df_immi_birth_place[df_immi_birth_place["Birthplace"] == birthplace]
    fig = px.choropleth_map(
        df_filtered,
        geojson=geojson_data,
        locations="DGUID",
        featureidkey="properties.DGUID",
        color="Proportion",
        color_continuous_scale="OrRd",
        hover_name="GEO",
        hover_data={"DGUID": False, "Proportion": True, "Count": True},
        map_style="open-street-map",
        center={"lat": 56, "lon": -106},
        zoom=3,
        opacity=0.7
    )
    return fig

# Build a Dash app
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='birthplace-dropdown',
        options=[{'label': bp, 'value': bp} for bp in unique_birthplaces],
        value=unique_birthplaces[0]
    ),
    dcc.Graph(id='map-graph')
])

@app.callback(
    Output('map-graph', 'figure'),
    Input('birthplace-dropdown', 'value')
)
def update_map(selected_birthplace):
    return create_map(selected_birthplace)

if __name__ == '__main__':
    app.run_server(debug=True, port=8051)


In [28]:
import geopandas as gpd
gdf_pr = gpd.read_file("../data/raw/geojson/lpr_000b21a_e_simplified_5percent.geojson")
gdf_pr.crs = "EPSG:3347"
gdf_pr = gdf_pr.to_crs(epsg=4326)
gdf_pr["geometry"] = gdf_pr["geometry"].buffer(0)
gdf_pr = gdf_pr[~gdf_pr.geometry.is_empty & gdf_pr.geometry.notnull()].copy()

In [31]:
gdf_pr.rename(columns={"PRENAME": "ADMIN"}, inplace=True)
gdf_pr = gdf_pr[["ADMIN", "geometry"]]

In [32]:
world_gdf = gpd.read_file("../data/processed/geojson/world_countries_clean.geojson")
world_gdf["geometry"] = world_gdf["geometry"].buffer(0)

In [33]:
world_gdf

,ADMIN,geometry
0,Zimbabwe,"POLYGON ((31.28789 -22.40205, 31.19727 -22.344..."
1,Zambia,"POLYGON ((30.39609 -15.64307, 30.25068 -15.643..."
2,Yemen,"MULTIPOLYGON (((53.76318 12.63682, 53.8248 12...."
3,Viet Nam,"MULTIPOLYGON (((107.97266 21.50796, 107.92578 ..."
4,Venezuela,"MULTIPOLYGON (((-60.01753 8.54932, -59.83164 8..."
...,...,...
237,Afghanistan,"POLYGON ((66.52227 37.34849, 66.82773 37.37129..."
238,Siachen Glacier,"POLYGON ((77.04863 35.10991, 77.00449 35.19634..."
239,Antarctica,"MULTIPOLYGON (((-57.02065 -63.37285, -56.92734..."
240,Sint Maarten (Dutch part),"POLYGON ((-63.12305 18.06895, -63.01118 18.068..."


In [34]:
import pandas as pd
world_gdf = pd.concat([world_gdf, gdf_pr])

In [35]:
world_gdf

,ADMIN,geometry
0,Zimbabwe,"POLYGON ((31.28789 -22.40205, 31.19727 -22.344..."
1,Zambia,"POLYGON ((30.39609 -15.64307, 30.25068 -15.643..."
2,Yemen,"MULTIPOLYGON (((53.76318 12.63682, 53.8248 12...."
3,Viet Nam,"MULTIPOLYGON (((107.97266 21.50796, 107.92578 ..."
4,Venezuela,"MULTIPOLYGON (((-60.01753 8.54932, -59.83164 8..."
...,...,...
8,Alberta,"POLYGON ((-110.00501 48.9997, -110.07314 48.99..."
9,British Columbia,"MULTIPOLYGON (((-123.78932 60.00003, -123.75 6..."
10,Yukon,"MULTIPOLYGON (((-136.4686 68.86953, -136.46741..."
11,Northwest Territories,"MULTIPOLYGON (((-123.38957 69.45706, -123.3891..."
